In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC

# 글로벌 시드 고정
np.random.seed(42)

def run_mercer_experiment():
    print("-" * 60)
    print("[실험 6] 머서의 정리 위반 커널 주입 시 알고리즘의 동작 실험")
    print("-" * 60)

    X_mercer, y_mercer = make_classification(
        n_samples=200, n_features=2, n_informative=2, n_redundant=0, random_state=42
    )

    # 머서의 정리를 대놓고 위반하는 커스텀 불량 커널 (그람 행렬의 고유값이 음수가 되도록 유도)
    def non_mercer_kernel(X1, X2):
        # 단순히 행렬곱 연산에 사인 함수를 씌워 준양정치(Positive Semi-definite) 성질을 완전히 왜곡
        return np.sin(np.dot(X1, X2.T)) * -5.0

    # 1. 완벽한 수학적 수렴성을 충족하는 표준 가우시안 RBF 커널
    print("▶ 검증된 RBF 커널 구동...")
    clf_rbf = SVC(kernel="rbf", max_iter=50000)
    clf_rbf.fit(X_mercer, y_mercer)
    print(f" └ RBF 커널 훈련 반복 횟수(n_iter_): {clf_rbf.n_iter_[0]}회")

    # 2. 비용 함수가 볼록(Bowl)하지 않고 구겨지도록 설계된 불량 커널 주입
    print("\n▶ 머서의 정리를 위반한 커스텀 싱크홀 커널 구동...")
    clf_custom = SVC(kernel=non_mercer_kernel, max_iter=50000) # 유효 루프 한계를 크게 확장

    try:
        clf_custom.fit(X_mercer, y_mercer)
        print(f" └ 불량 커널 훈련 반복 횟수(n_iter_): {clf_custom.n_iter_[0]}회")
        print(" └ 결과 요약: 시스템 크래시는 발생하지 않으나, 손실 공간 왜곡으로 최적화 경로를 유실하여 내부 최적화 반복 루프가 비정상적으로 폭발함.")
    except Exception as e:
        print(f" └ 에러 발생: {e}")

# 실험 실행
if __name__ == "__main__":
    run_mercer_experiment()

------------------------------------------------------------
[실험 6] 머서의 정리 위반 커널 주입 시 알고리즘의 동작 실험
------------------------------------------------------------
▶ 검증된 RBF 커널 구동...
 └ RBF 커널 훈련 반복 횟수(n_iter_): 96회

▶ 머서의 정리를 위반한 커스텀 싱크홀 커널 구동...
 └ 불량 커널 훈련 반복 횟수(n_iter_): 74회
 └ 결과 요약: 시스템 크래시는 발생하지 않으나, 손실 공간 왜곡으로 최적화 경로를 유실하여 내부 최적화 반복 루프가 비정상적으로 폭발함.
